### Задание 1
Загрузите данные и определите разницу в конверсиях между контрольной и тестовой группами. Сгруппируйте данные по колонке treatment и вычислите среднее значение и количество для столбца conversion.

In [1]:
# подготовка данных и обучение модели
import pandas as pd

# загрузите данные
data = pd.read_csv("yandex_plus.csv")

# по treatment: средняя конверсия и размер каждой группы
conv = data.groupby('treatment')['conversion'].agg(['mean', 'count'])

print(conv)

              mean  count
treatment                
control     0.4056   5000
treatment1  0.4932   5000


### Задание 2
Какая разница в конверсии между контрольной и тестовой группами?

In [4]:
# разница средних конверсий: treatment1 − control
diff_conv = conv.loc['treatment1', 'mean'] - conv.loc['control', 'mean']
print(round(diff_conv, 2))  # округление до двух знаков после запятой

0.09


### Задание 3
Обучите алгоритм uplift-деревьев, предварительно разделив выборку на тренировочную и валидационную в соотношении 70 на 30. Определите модель UpliftRandomForestClassifier, передав в аргументы параметры модели. Напишите алгоритм подбора оптимальных параметров модели на основе метрики Uplift@10%, которую необходимо максимизировать.
В этом задании вам нужно:
1. разделить данные на тренировочную (70%) и валидационную (30%) выборки;
2. создать и обучить модель UpliftRandomForestClassifier с различными параметрами;
3. оценить качество модели с помощью метрики Uplift@10% (эффект воздействия для топ-10% клиентов);
4. найти оптимальные параметры модели, которые максимизируют метрику Uplift@10%.

В модели UpliftRandomForestClassifier можно регулировать следующие параметры:
- evaluationFunction
  - функция оценки качества разбиения (KL, ED, Chi и др.);
- max_depth
  - максимальная глубина деревьев;
- n_estimators
  - количество деревьев в ансамбле.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from causalml.inference.tree import UpliftRandomForestClassifier
from itertools import product

# загружаем данные
data = pd.read_csv("yandex_plus.csv")

# разделение на признаки и целевую переменную
X = data.drop(['conversion'], axis=1)
y = data['conversion']

# разделение данных на обучающую и валидационную выборки (70% и 30%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3,  # 30% — валидация, 70% — обучение
    random_state=42
)

max_uplift = 0
top_percent = 0.1
best_params = {}

eval_functions = ['KL', 'ED', 'Chi']
max_depths = [6, 7, 8]
n_estimators = [50, 100]

results = []

for eval_f, m_depth, n_est in product(eval_functions, max_depths, n_estimators):
    uplift_model = UpliftRandomForestClassifier(
        control_name='control',
        evaluationFunction=eval_f,
        max_depth=m_depth,
        n_estimators=n_est,
        random_state=42
    )

    # обучите модель на обучающей выборке
    # признаки без treatment, отдельно метки групп и целевая переменная
    uplift_model.fit(
        X_train.drop(columns=['treatment']).values,
        treatment=X_train['treatment'].values,
        y=y_train.values
    )

    # получите предсказания uplift на валидационной выборке
    # колонку treatment в признаки не подаём
    uplift_preds = uplift_model.predict(
        X_val.drop(columns=['treatment']).values
    )

    # отсортируйте по убыванию
    uplift_preds_df = (pd.DataFrame(uplift_preds, columns = ['uplift_score'])
                ['uplift_score']
                .sort_values(ascending=False)  # сначала клиенты с наибольшим uplift
    ) 

    # рассчитайте средний uplift для топ-10% процентов пользователей
    mean_uplift = uplift_preds_df.head(int(len(uplift_preds_df) * top_percent)).mean()
    
    results.append({
        'eval_function': eval_f,
        'max_depth': m_depth,
        'n_estimators': n_est,
        'mean_uplift@10%': mean_uplift
    })

    # сравните текущее значение mean_uplift с текущим максимальным значением max_uplift
    # если mean_uplift > max_uplift, то перезапишите новое максимальное значение в переменную max_uplift
    # передайте в словарь best_params новые оптимальные параметры 
    
    # обновляем лучший результат, если текущий Uplift@10% выше
    if mean_uplift > max_uplift:
        max_uplift = mean_uplift
        best_params = {
            'eval_function': eval_f,
            'max_depth': m_depth,
            'n_estimators': n_est
        }

# преобразуйте результаты в DataFrame и выведите лучшие параметры
results_df = pd.DataFrame(results)

Best params: {'eval_function': 'ED', 'max_depth': 8, 'n_estimators': 100}
Best uplift@10%: 0.29201481331494156


### Задание 4
Напечатать оптимальные параметры в формате

пример:
eval_function='ED', max_depth=8, n_estimators=100.


In [6]:
print("Best params:", best_params)
print("Best uplift@10%:", max_uplift)

Best params: {'eval_function': 'ED', 'max_depth': 8, 'n_estimators': 100}
Best uplift@10%: 0.29201481331494156


### Задание 5
Переобучите модель с оптимальными параметрами, полученными в прошлом задании, и сделайте прогноз на валидационной выборке.

In [ ]:
# инициализация модели
# оптимальные параметры с задания 3: ED, глубина 8, 100 деревьев
uplift_model = UpliftRandomForestClassifier(
        control_name='control',
        evaluationFunction='ED',
        max_depth=8,
        n_estimators=100,
        random_state=42
    )
# обучение модели
uplift_model.fit(
        X_train.drop(columns=['treatment']).values,
        treatment=X_train['treatment'].values,
        y=y_train.values
    )
# получение прогнозов
uplift_preds = uplift_model.predict(X_val.drop(columns=['treatment']), full_output=False)


### Задание 6
Выберите пользователей со значением uplift больше 0.2. Для удобства прогнозы были переведены в формат pandas.DataFrame с единственной колонкой uplift_score. Сохраните полученные значения в CSV-формате с названием файла top_users.csv.

In [8]:

# прогнозы задания 5 — DataFrame с колонкой uplift_score
uplift_preds_df = pd.DataFrame(uplift_preds, columns=['uplift_score'])

# пользователи с предсказанным uplift выше 0.2
top_users = uplift_preds_df[uplift_preds_df['uplift_score'] > 0.2]

# сохраните полученный датафрейм
top_users.to_csv('top_users.csv', index=False) 